<a href="https://colab.research.google.com/github/Sathyarajpalle/NLP-practice/blob/main/FakenewsclassifierusingLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fake news classifier using LSTM**

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('/content/drive/MyDrive/fake_train.csv')

In [4]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [5]:
df.shape

(20800, 5)

In [6]:
df.isnull().sum()

,0
id,0
title,558
author,1957
text,39
label,0


In [7]:
##drop nan values
df = df.dropna()

In [8]:
df.shape

(18285, 5)

In [10]:
##get the independent features
x = df.drop('label', axis = 1)

In [9]:
## get the dependent features
y = df['label']

In [12]:
x.shape

(18285, 4)

In [13]:
y.shape

(18285,)

In [14]:
import tensorflow as tf

In [15]:
tf.__version__

'2.20.0'

In [16]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [17]:
### vocubulary size
voc_size = 5000

In [18]:
messages = x.copy()

In [19]:
messages['title'][1]

'FLYNN: Hillary Clinton, Big Woman on Campus - Breitbart'

In [20]:
messages

,id,title,author,text
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [22]:
import nltk
import re
from nltk.corpus import stopwords

In [23]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [30]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()
# Reset the index to ensure sequential integer indexing after dropping rows
#messages.reset_index(drop=True, inplace=True)
corpus = []
for i in range(0, len(messages)):
  review = re.sub('[^a-zA-Z0-9]',' ',messages['title'][i])
  review = review.lower()
  review = review.split()
  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

In [31]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 '15 civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay 2 5 million releas everyon arrest dakota access pipelin',
 'fbi clos

In [33]:
corpus[1]

'flynn hillari clinton big woman campu breitbart'

In [32]:
onehot_repr = [one_hot(words, voc_size) for words in corpus]
onehot_repr

[[913, 2103, 66, 1549, 2796, 2625, 1693, 250, 4987, 2654],
 [730, 1290, 2224, 750, 92, 2872, 1182],
 [4032, 1368, 1385, 415],
 [3044, 3096, 970, 179, 3453, 781, 110],
 [1862, 92, 4192, 4942, 3680, 4555, 92, 4184, 2968, 1804],
 [4597,
  4214,
  2681,
  3662,
  4134,
  4222,
  1438,
  2285,
  3394,
  1813,
  2703,
  407,
  4398,
  3687,
  1182],
 [945, 4118, 1027, 3739, 3562, 292, 3949, 3929, 3783, 1017, 721],
 [3805, 886, 3867, 2272, 2733, 4643, 4222, 3460, 3783, 1017, 721],
 [4669, 727, 4725, 3984, 813, 1839, 3812, 1002, 4222, 2854],
 [4602, 1876, 4671, 434, 4991, 4614, 3757, 936],
 [564, 1162, 1990, 3138, 675, 4505, 2783, 576, 271, 696, 3749],
 [3453, 4856, 2796, 1839, 4222, 2733],
 [4894, 4079, 1378, 1720, 3260, 3055, 3502, 644, 355],
 [4632, 1689, 279, 2711, 231, 622, 3993, 3783, 1017, 721],
 [3306, 1252, 1213, 496, 3715, 3783, 1017, 721],
 [2428, 2157, 4048, 286, 3600, 732, 4454, 798, 174, 2418, 1608, 2157],
 [1829, 1277, 1290],
 [4434, 2091, 1875, 1997, 4222, 1295, 1508, 1182],
 [

In [34]:
corpus[1]

'flynn hillari clinton big woman campu breitbart'

In [35]:
onehot_repr[1]

[730, 1290, 2224, 750, 92, 2872, 1182]

**Embedding Representation**

In [36]:
sent_length = 20
embedded_docs = pad_sequences(onehot_repr, padding = 'pre', maxlen = sent_length)
print(embedded_docs)

[[   0    0    0 ...  250 4987 2654]
 [   0    0    0 ...   92 2872 1182]
 [   0    0    0 ... 1368 1385  415]
 ...
 [   0    0    0 ... 3783 1017  721]
 [   0    0    0 ... 1596 2831 3595]
 [   0    0    0 ... 2855   30 3894]]


In [38]:
## Creating model
embedding_vector_features=40 ##features representation
model=Sequential()
model.add(Embedding(input_dim = voc_size,output_dim = embedding_vector_features,input_shape = (sent_length,)))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

None


In [39]:
len(embedded_docs),y.shape

(18285, (18285,))

In [43]:
import numpy as np
x_final = np.array(embedded_docs)
y_final = np.array(y)

In [44]:
x_final.shape,y_final.shape

((18285, 20), (18285,))

In [47]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_final, y_final, test_size=0.33, random_state=42)

**Model Training**

In [48]:
### Finally Training
model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.9996 - loss: 0.0021 - val_accuracy: 0.9122 - val_loss: 0.6217
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.9997 - loss: 0.0016 - val_accuracy: 0.9137 - val_loss: 0.6495
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.9996 - loss: 0.0017 - val_accuracy: 0.9102 - val_loss: 0.7266
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 11s 39ms/step - accuracy: 0.9998 - loss: 9.9776e-04 - val_accuracy: 0.9147 - val_loss: 0.6353
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 1.0000 - loss: 4.6996e-04 - val_accuracy: 0.9118 - val_loss: 0.6964
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.9999 - loss: 2.6473e-04 - val_accuracy: 0.9100 - val_loss: 0.7032
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 1.0000 - loss: 2.0826e-04 - val_accuracy: 0.9104 - val_loss: 0.7684
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 1.0000 - loss

**Prediction and accuracy**

In [51]:
y_pred = (model.predict(x_test) > 0.5).astype(int)

189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [52]:
y_pred = np.where(y_pred > 0.5, 1, 0)

In [53]:
from sklearn.metrics import confusion_matrix

In [54]:
confusion_matrix(y_test,y_pred)

array([[3127,  292],
       [ 247, 2369]])

In [55]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9106876553438277

In [69]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.57      1.00      0.72      3419
           1       0.00      0.00      0.00      2616

    accuracy                           0.57      6035
   macro avg       0.28      0.50      0.36      6035
weighted avg       0.32      0.57      0.41      6035



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
